In [8]:
import os
HOME = os.getcwd()
print(HOME)

/content


# 1. Instalar librerias

In [9]:
!pip install supervision
!pip install ultralytics
!pip install trackers
!pip install gdown
!pip install tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.7/224.7 kB 10.2 MB/s eta 0:00:00


# 2. Descargar video y pesos del modelo

In [10]:
# Get the directory where the script is located
!mkdir -p $HOME/data

# Download the traffic_analysis.mov file from Google Drive
!gdown -O $HOME/data/traffic_analysis.mov "https://drive.google.com/uc?id=1qadBd7lgpediafCpL_yedGjQPk-FLK-W"

# Download the traffic_analysis.pt file from Google Drive
!gdown -O $HOME/data/traffic_analysis.pt "https://drive.google.com/uc?id=1y-IfToCjRXa3ZdC1JpnKRopC7mcQW-5z"

Downloading...
From (original): https://drive.google.com/uc?id=1qadBd7lgpediafCpL_yedGjQPk-FLK-W
From (redirected): https://drive.google.com/uc?id=1qadBd7lgpediafCpL_yedGjQPk-FLK-W&confirm=t&uuid=981e93c7-c5e1-40ad-ae93-a1d4978fcf09
To: /content/data/traffic_analysis.mov
100% 349M/349M [00:04<00:00, 78.5MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1y-IfToCjRXa3ZdC1JpnKRopC7mcQW-5z
From (redirected): https://drive.google.com/uc?id=1y-IfToCjRXa3ZdC1JpnKRopC7mcQW-5z&confirm=t&uuid=da1b5801-4db4-4ca2-849e-2bb4572980bb
To: /content/data/traffic_analysis.pt
100% 137M/137M [00:01<00:00, 119MB/s]


# 3. Importar librerias

In [11]:
from typing import Dict, List, Set, Tuple
from trackers import ByteTrackTracker
from ultralytics import YOLO
from tqdm import tqdm

import supervision as sv
import numpy as np

a. Detections Manager

In [12]:
class DetectionsManager:
    def __init__(self) -> None:
        self.tracker_id_to_zone_id: Dict[int, int] = {}
        self.counts: Dict[int, Dict[int, Set[int]]] = {}

    def update(
        self,
        detections_all: sv.Detections,
        detections_in_zones: List[sv.Detections],
        detections_out_zones: List[sv.Detections],
    ) -> sv.Detections:
        for zone_in_id, detections_in_zone in enumerate(detections_in_zones):
            for tracker_id in detections_in_zone.tracker_id:
                self.tracker_id_to_zone_id.setdefault(tracker_id, zone_in_id)

        for zone_out_id, detections_out_zone in enumerate(detections_out_zones):
            for tracker_id in detections_out_zone.tracker_id:
                if tracker_id in self.tracker_id_to_zone_id:
                    zone_in_id = self.tracker_id_to_zone_id[tracker_id]
                    self.counts.setdefault(zone_out_id, {})
                    self.counts[zone_out_id].setdefault(zone_in_id, set())
                    self.counts[zone_out_id][zone_in_id].add(tracker_id)

        detections_all.class_id = np.vectorize(
            lambda x: self.tracker_id_to_zone_id.get(x, -1)
        )(detections_all.tracker_id)
        return detections_all[detections_all.class_id != -1]

b. Inizializar poligonos

In [13]:
ZONE_IN_POLYGONS = [
    np.array([[592, 282], [900, 282], [900, 82], [592, 82]]),
    np.array([[950, 860], [1250, 860], [1250, 1060], [950, 1060]]),
    np.array([[592, 582], [592, 860], [392, 860], [392, 582]]),
    np.array([[1250, 282], [1250, 530], [1450, 530], [1450, 282]]),
]

ZONE_OUT_POLYGONS = [
    np.array([[950, 282], [1250, 282], [1250, 82], [950, 82]]),
    np.array([[592, 860], [900, 860], [900, 1060], [592, 1060]]),
    np.array([[592, 282], [592, 550], [392, 550], [392, 282]]),
    np.array([[1250, 860], [1250, 560], [1450, 560], [1450, 860]]),
]

def initiate_polygon_zones(
    polygons: List[np.ndarray],
    triggering_anchors: List[sv.Position] = [sv.Position.CENTER],
) -> List[sv.PolygonZone]:
    return [
        sv.PolygonZone(
            polygon=polygon,
            triggering_anchors=triggering_anchors,
        )
        for polygon in polygons
    ]

c. Procesar video

In [14]:
from typing import Dict, List, Set, Tuple
from ultralytics import YOLO
from tqdm import tqdm

import supervision as sv
import numpy as np

# Re-defining the class with the correction
COLORS = sv.ColorPalette.DEFAULT

class VideoProcessor:
    def __init__(
        self,
        source_weights_path: str,
        source_video_path: str,
        target_video_path: str = None,
        confidence_threshold: float = 0.3,
        iou_threshold: float = 0.7,
    ) -> None:
        self.conf_threshold = confidence_threshold
        self.iou_threshold = iou_threshold
        self.source_video_path = source_video_path
        self.target_video_path = target_video_path

        self.model = YOLO(source_weights_path)
        self.tracker = ByteTrackTracker()
        self.video_info = sv.VideoInfo.from_video_path(source_video_path)

        # Assuming ZONE_IN_POLYGONS and ZONE_OUT_POLYGONS are already defined and available in the global scope.
        # Assuming initiate_polygon_zones is already defined and available in the global scope.
        self.zones_in = initiate_polygon_zones(ZONE_IN_POLYGONS)
        self.zones_out = initiate_polygon_zones(ZONE_OUT_POLYGONS)

        # Correction applied here: sv.BoundingBoxAnnotator changed to sv.BoxAnnotator
        self.boundingBoxAnnotator = sv.BoxAnnotator(
            color=COLORS, thickness=2, color_lookup=sv.ColorLookup.CLASS
        )
        self.trace_annotator = sv.TraceAnnotator(
            color=COLORS, position=sv.Position.CENTER, trace_length=100, thickness=2
        )
        self.detections_manager = DetectionsManager()

    def process_video(self):
        frame_generator = sv.get_video_frames_generator(
            source_path=self.source_video_path
        )

        with sv.VideoSink(self.target_video_path, self.video_info) as sink:
          for frame in tqdm(frame_generator, total=self.video_info.total_frames):
            annotated_frame = self.process_frame(frame)
            sink.write_frame(annotated_frame)

    def annotate_frame(
        self, frame: np.ndarray, detections: sv.Detections
    ) -> np.ndarray:
        annotated_frame = frame.copy()
        for i, (zone_in, zone_out) in enumerate(zip(self.zones_in, self.zones_out)):
            annotated_frame = sv.draw_polygon(
                annotated_frame, zone_in.polygon, COLORS.colors[i]
            )
            annotated_frame = sv.draw_polygon(
                annotated_frame, zone_out.polygon, COLORS.colors[i]
            )

        labels = [f"#{tracker_id}" for tracker_id in detections.tracker_id]
        annotated_frame = self.trace_annotator.annotate(annotated_frame, detections)
        annotated_frame = self.boundingBoxAnnotator.annotate(
            annotated_frame, detections
        )

        for zone_out_id, zone_out in enumerate(self.zones_out):
            zone_center = sv.get_polygon_center(polygon=zone_out.polygon)
            if zone_out_id in self.detections_manager.counts:
                counts = self.detections_manager.counts[zone_out_id]
                for i, zone_in_id in enumerate(counts):
                    count = len(self.detections_manager.counts[zone_out_id][zone_in_id])
                    text_anchor = sv.Point(x=zone_center.x, y=zone_center.y + 40 * i)
                    annotated_frame = sv.draw_text(
                        scene=annotated_frame,
                        text=str(count),
                        text_anchor=text_anchor,
                        background_color=COLORS.colors[zone_in_id],
                    )

        return annotated_frame

    def process_frame(self, frame: np.ndarray) -> np.ndarray:
        results = self.model(
            frame, verbose=False, conf=self.conf_threshold, iou=self.iou_threshold
        )[0]
        detections = sv.Detections.from_ultralytics(results)
        detections.class_id = np.zeros(len(detections))
        detections = self.tracker.update(detections)

        detections_in_zones = []
        detections_out_zones = []

        for i, (zone_in, zone_out) in enumerate(zip(self.zones_in, self.zones_out)):
            detections_in_zone = detections[zone_in.trigger(detections=detections)]
            detections_in_zones.append(detections_in_zone)
            detections_out_zone = detections[zone_out.trigger(detections=detections)]
            detections_out_zones.append(detections_out_zone)

        detections = self.detections_manager.update(
            detections, detections_in_zones, detections_out_zones
        )
        return self.annotate_frame(frame, detections)

# Re-instantiate and run the processor
processor = VideoProcessor(
      source_weights_path="data/traffic_analysis.pt",
      source_video_path="data/traffic_analysis.mov",
      target_video_path="traffic_analysis_result.mov",
      confidence_threshold=0.3,
      iou_threshold=0.5,
  )
processor.process_video()

100%|██████████| 806/806 [01:36<00:00,  8.39it/s]
